In [1]:
import re, unicodedata
from collections import Counter
import numpy as np
import pandas as pd
import networkx as nx
import networkx.algorithms.community as nx_comm
from difflib import SequenceMatcher
from sentence_transformers import SentenceTransformer

# --- 1. Configuration (Frozen Validated Baseline) ---
INPUT_CSV = "qc_applicants_PROD.csv"      # <-- replace with your use-case filename
OUTPUT_DATA_CSV = "harmonized_output.csv" # original table + harm_id + harmonized_name
OUTPUT_MAP_CSV = "harmonization_lookup.csv" # clean 1-to-1 dictionary (raw_name -> harmonized_name)
NAME_COL = "psn_name"                  # <-- column containing raw applicant names
WEIGHT_COL = None                         # optional: e.g. "n_families" to weight canonical name choice

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
SCORE_WEIGHTS = {"cosine": 0.34, "jaccard": 0.33, "edit": 0.33}
SCORE_FLOOR = 0.45
MIN_OTSU_FLOOR = 0.56 #avoid expressions that lower it to the ground
STOPWORD_DF_RATIO = 0.008 #give an explaination 
LOUVAIN_RESOLUTION = 1.5
LOUVAIN_SEED = 42

# --- 2. Text Normalization (with full-word legal forms added) ---
FORMS = [
    "INC", "INCORPORATED", "LLC", "LLP", "LP", "LTD", "LIMITED", "CORP", "CORPORATION",
    "CO", "COMPANY", "PLC", "GMBH", "MBH", "AG", "KG", "SE", "SA", "SAS", "SARL", "SRL",
    "SPA", "SL", "BV", "NV", "OY", "OYJ", "AB", "AS", "ASA", "APS", "PTY", "PTE", "KK",
    "KABUSHIKI KAISHA", "SP Z OO", "OOO", "BHD", "SDN", "PVT", "LTDA", "LIMITEE", "LTEE",
    "AKTIENGESELLSCHAFT", "SOCIETE ANONYME", "SOCIEDAD ANONIMA", "SOCIETA PER AZIONI"
]
FORMS_RE = "|".join(sorted(FORMS, key=len, reverse=True))
FORMS_CJK = "股份有限公司|有限责任公司|有限責任公司|有限公司|株式会社|有限会社|合同会社|주식회사|公司"
WORDS = {
    "UNIVERSITY": "UNIV", "UNIVERSITAET": "UNIV", "UNIVERSITAT": "UNIV", "UNIVERSITE": "UNIV",
    "UNIVERSIDAD": "UNIV", "UNIVERSITA": "UNIV", "INSTITUTE": "INST", "INSTITUT": "INST",
    "INSTITUTO": "INST", "TECHNOLOGY": "TECH", "TECHNOLOGIES": "TECH", "TECHNOLOGICAL": "TECH",
    "LABORATORY": "LAB", "LABORATORIES": "LAB", "LABS": "LAB", "CENTRE": "CENTER",
    "SCIENCES": "SCIENCE", "大學": "大学"
}
FILLER = {"OF", "THE", "FOR", "AND", "DE", "DI", "DU", "DES", "ET", "UND"}

def clean(name):
    if pd.isna(name):
        return ""
    s = unicodedata.normalize("NFKD", unicodedata.normalize("NFKC", str(name)).upper())
    s = unicodedata.normalize("NFC", "".join(c for i, c in enumerate(s) if not (unicodedata.combining(c) and s[i - 1] < "ɐ")))
    s = re.sub(r"[▲▼?.'’]", "", s)
    latin = re.search(r"[^\x00-\x7f]([A-Z0-9][A-Z0-9 ,&\-/]+)$", s)
    if latin and len(re.sub(r"[^A-Z]", "", latin.group(1))) >= 3:
        s = latin.group(1)
    s = re.sub(r"^C/O\s+", "", s).strip()
    s = re.sub(r"(?:,|\s)\s*\bC/O\b.*$", "", s)
    s = re.sub(r"\(\s*A\s+[A-Z ]+\b(?:CORP|CORPORATION|COMPANY|LIMITED|LTD|INC)\b[^)]*\)", " ", s)
    s = s.replace("A/S", "AS").replace("&", " AND ")
    s = re.sub(rf"^({FORMS_CJK})|({FORMS_CJK})$", "", re.sub(r"[^\w ]|_", " ", s).strip())
    s = " ".join(WORDS.get(w, w) for w in s.split() if w not in FILLER) or s
    s = re.sub(rf"^(({FORMS_RE}) )+|( ({FORMS_RE}))+$", "", s).strip() or s
    return s

# --- 3. Sector-Adaptive Pairwise Scoring (IDF-Jaccard + Full-String Edit) ---
def pair_features_sector(names, emb, w=SCORE_WEIGHTS, floor=SCORE_FLOOR):
    n = len(names)
    wc, wj, we = w["cosine"], w["jaccard"], w["edit"]
    tokens = [set(s.split()) for s in names]
    lens = np.array([len(s) for s in names], dtype=np.float32)
    # Corpus-driven IDF weights (penalizes QUANTUM, TECH, UNIV, STATE, GRID, etc.) STILL NOT ENOUGH
    df_counts = Counter(t for toks in tokens for t in toks)
    idf = {t: np.log(n / c) for t, c in df_counts.items()}
    tok_idf_sums = np.array([sum(idf[t] for t in toks) for toks in tokens], dtype=np.float32)
    sim_matrix = emb @ emb.T
    min_cos = (floor - wj - we) / wc
    rows, cols = np.where(np.triu(sim_matrix >= min_cos, k=1))
    out = []
    for i, j in zip(rows, cols):
        cos = float(sim_matrix[i, j])
        ta, tb = tokens[i], tokens[j]
        inter = ta & tb
        if inter:
            inter_idf = sum(idf[t] for t in inter)
            union_idf = tok_idf_sums[i] + tok_idf_sums[j] - inter_idf
            jac = float(inter_idf / union_idf) if union_idf > 0 else 0.0
        else:
            jac = 0.0
        if wc * cos + wj * jac + we < floor:
            continue
        li, lj = lens[i], lens[j]
        max_ed = (2.0 * min(li, lj) / (li + lj)) if (li + lj) > 0 else 0.0
        if wc * cos + wj * jac + we * max_ed < floor:
            continue
        ed = SequenceMatcher(None, names[i], names[j]).ratio()
        # Require either shared token overlap, high surface spelling similarity, or near-exact acronym match
        if jac == 0.0 and ed < 0.78 and cos < 0.92:
            continue
        score = wc * cos + wj * jac + we * ed
        if score >= floor:
            out.append((names[i], names[j], score))
    return pd.DataFrame(out, columns=["name_a", "name_b", "score"])

def find_threshold_otsu(scores, n_bins=100):
    counts, bin_edges = np.histogram(scores, bins=n_bins)
    centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    w1 = np.cumsum(counts)
    w2 = w1[-1] - w1
    valid = (w1 > 0) & (w2 > 0)
    w1, w2, centers_v = w1[valid], w2[valid], centers[valid]
    csum = np.cumsum(counts * centers)[valid]
    total_sum = np.sum(counts * centers)
    m1 = csum / w1
    m2 = (total_sum - csum) / w2
    between_var = w1 * w2 * (m1 - m2) ** 2
    return float(centers_v[np.argmax(between_var)])

# --- 4. Run End-to-End Inference ---
df = pd.read_csv(INPUT_CSV)
df = df[df["psn_sector"] != "INDIVIDUAL"] # removes individuals from harmonization
df["clean"] = df[NAME_COL].apply(clean)
valid_mask = df["clean"].str.len() > 0
unique_names = sorted(df.loc[valid_mask, "clean"].unique())
print(f"Loaded {len(df):,} rows | {len(unique_names):,} unique non-empty cleaned names")

model = SentenceTransformer(MODEL_NAME)
emb = model.encode(unique_names, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)

scored = pair_features_sector(unique_names, emb)
raw_otsu = find_threshold_otsu(scored["score"].values) if len(scored) > 0 else MIN_OTSU_FLOOR
th_otsu = max(raw_otsu, MIN_OTSU_FLOOR)
print(f"Kept {len(scored):,} candidate edges | Unsupervised Otsu threshold: {th_otsu:.3f}")

G = nx.Graph()
G.add_nodes_from(unique_names)
edges = scored[scored["score"] >= th_otsu]
G.add_weighted_edges_from(zip(edges["name_a"], edges["name_b"], edges["score"]))
communities = nx_comm.louvain_communities(G, weight="weight", resolution=LOUVAIN_RESOLUTION, seed=LOUVAIN_SEED)

# Rank canonical names by frequency (or patent weight if WEIGHT_COL is provided)
if WEIGHT_COL and WEIGHT_COL in df.columns:
    name_weights = df.groupby("clean")[WEIGHT_COL].sum().to_dict()
else:
    name_weights = df["clean"].value_counts().to_dict()

# Sort communities deterministically so harm_id (HARM_00001, ...) is reproducible
sorted_comms = sorted(
    communities,
    key=lambda c: (-sum(name_weights.get(m, 0) for m in c), min(c))
)

clean_to_canon = {}
clean_to_id = {}
for idx, members in enumerate(sorted_comms, start=1):
    harm_id = f"HARM_{idx:05d}"
    best_name = min(members, key=lambda n: (-name_weights.get(n, 0), len(n), n))
    for m in members:
        clean_to_canon[m] = best_name
        clean_to_id[m] = harm_id

df["harm_id"] = df["clean"].map(clean_to_id)
df["harmonized_name"] = df["clean"].map(clean_to_canon)

# --- 5. Export Deliverables + Quick QA Summary ---
df.to_csv(OUTPUT_DATA_CSV, index=False)

lookup_df = (
    df[[NAME_COL, "clean", "harm_id", "harmonized_name"]]
    .drop_duplicates()
    .sort_values(["harm_id", NAME_COL])
)
lookup_df.to_csv(OUTPUT_MAP_CSV, index=False)

print(f"Harmonized {len(unique_names):,} distinct names into {len(sorted_comms):,} entities "
      f"({100 * (1 - len(sorted_comms) / max(len(unique_names), 1)):.1f}% consolidation).")
print(f"Saved full dataset -> {OUTPUT_DATA_CSV}")
print(f"Saved lookup table -> {OUTPUT_MAP_CSV}")
print("\nTop 10 largest harmonized entities:")
print(df["harmonized_name"].value_counts().head(10))

Loaded 18,812 rows | 3,375 unique non-empty cleaned names


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/106 [00:00<?, ?it/s]

Kept 27,293 candidate edges | Unsupervised Otsu threshold: 0.565
Harmonized 3,375 distinct names into 1,997 entities (40.8% consolidation).
Saved full dataset -> harmonized_output.csv
Saved lookup table -> harmonization_lookup.csv

Top 10 largest harmonized entities:
harmonized_name
ORIGIN QUANTUM COMPUTING TECH HEFEI                       1303
RUBAN QUANTUM TECH                                         844
IBM INTERNATIONAL BUSINESS MACHINES                        797
BEIJING BAIDU NETCOM SCIENCE TECH                          560
TSINGHUA UNIV                                              510
TENCENT TECH SHENZHEN                                      483
MICROSOFT TECH LICENSING                                   287
IBM DEUTSCHLAND                                            282
SHANGHAI TAILINGWISHI CALCULATION QUANTUM SCIENCE TECH     276
QUANTINUUM                                                 273
Name: count, dtype: int64
